# 04c Complete-case sensitivity network

**Purpose.** Evaluate whether the primary median-imputed Graphical LASSO network (n=1,286) is robust to the strategy used for handling missing laboratory values, by reconstructing the network using only participants with **no missing values** among the 29 primary biomarkers (complete-case analysis), and comparing it against the primary network.

This notebook is fully self-contained and reproduces, on the complete-case subset, exactly the same preprocessing (winsorizing / log-transform already applied upstream in stage 02, z-scoring), Graphical LASSO EBIC(gamma=0.5) + density<=0.20 model-selection rule, and centrality / bridge-centrality formulas used for the primary network (stages 02, 04, 05).

Run after stages 00-05 have been executed (it reads their saved outputs as inputs).

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import GraphicalLasso
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

PROJECT_ROOT = Path('/content/pcos_network_architecture_jcem_v3_jcem_10of10') if Path('/content/pcos_network_architecture_jcem_v3_jcem_10of10').exists() else Path.cwd().parent
print('Project root:', PROJECT_ROOT)

OUT = PROJECT_ROOT / 'outputs' / '04c_complete_case_network'
OUT.mkdir(parents=True, exist_ok=True)


## 1. Load the primary (winsorized / log-transformed, **not imputed**) biomarker matrix

`primary_direct_features_unimputed.csv` (stage 02) contains the same 29 primary biomarkers after outlier winsorizing and skew-correcting log-transform as the primary imputed dataset, but **before** median imputation — i.e. missing values are still `NaN`. Restricting this table to rows with zero missing values across the 29 biomarkers gives the complete-case cohort.

In [ ]:
Y = pd.read_csv(PROJECT_ROOT / 'outputs' / '02_primary_network_dataset' / 'primary_direct_features_unimputed.csv')
feat = pd.read_csv(PROJECT_ROOT / 'outputs' / '02_primary_network_dataset' / 'primary_feature_set.csv')
DOM = dict(zip(feat.feature, feat.domain))
cols = list(feat.feature)

CC = Y.dropna(subset=cols).reset_index(drop=True)
n_complete = len(CC)
print(f'Complete-case n = {n_complete} / {len(Y)} ({100*n_complete/len(Y):.1f}% of the primary cohort)')

Z = pd.DataFrame(StandardScaler().fit_transform(CC[cols]), columns=cols)


## 2. Graphical LASSO model selection (identical rule to the primary network)

EBIC with gamma = 0.50, restricted to alpha values whose resulting network density satisfies the pre-specified sparse-interpretability constraint (density <= 0.20); among eligible alpha values the one minimizing EBIC is selected — exactly the rule documented for the primary network (`primary_graphical_lasso_model_summary.json`).

In [ ]:
def edge_df_from_partial(P, cols, threshold=1e-8):
    rows = []
    for i, a in enumerate(cols):
        for j, b in enumerate(cols):
            if j <= i:
                continue
            w = P[i, j]
            if abs(w) > threshold:
                rows.append({'source': a, 'target': b, 'weight': float(w),
                             'abs_weight': float(abs(w)), 'sign': 'positive' if w > 0 else 'negative'})
    return pd.DataFrame(rows).sort_values('abs_weight', ascending=False) if rows else \
        pd.DataFrame(columns=['source', 'target', 'weight', 'abs_weight', 'sign'])


def select_glasso_ebic_density_constrained(Z, alphas=None, gamma=0.5, max_density=0.20):
    if alphas is None:
        alphas = [0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60]
    n, p = Z.shape
    rows, fitted = [], {}
    for a in alphas:
        try:
            m = GraphicalLasso(alpha=a, max_iter=150, tol=1e-3).fit(Z)
            prec = m.precision_
            off = (np.abs(prec) > 1e-8)
            np.fill_diagonal(off, False)
            e = int(off.sum() / 2)
            density = e / (p * (p - 1) / 2)
            score = float(m.score(Z))
            ebic = float(-2 * n * score + e * np.log(n) + 4 * gamma * e * np.log(p))
            rows.append({'alpha': a, 'log_likelihood_per_sample': score, 'n_edges': e,
                         'density': density, 'EBIC': ebic,
                         'eligible_density_le_0_20': bool(density <= max_density)})
            fitted[a] = m
        except Exception as ex:
            rows.append({'alpha': a, 'error': str(ex)})
    table = pd.DataFrame(rows)
    eligible = table[table.get('eligible_density_le_0_20', False) == True]
    if len(eligible) == 0:
        raise RuntimeError('No alpha satisfied the density <= 0.20 constraint.')
    best_row = eligible.loc[eligible['EBIC'].idxmin()]
    best_alpha = float(best_row['alpha'])
    return fitted[best_alpha], table, best_alpha


def graph_metrics(edges, domains):
    G = nx.Graph()
    for f, d in domains.items():
        G.add_node(f, domain=d)
    for _, r in edges.iterrows():
        G.add_edge(r.source, r.target, weight=float(r.weight), abs_weight=float(r.abs_weight))
    strength = {n: sum(d['abs_weight'] for _, _, d in G.edges(n, data=True)) for n in G.nodes}
    degree = dict(G.degree())
    between = nx.betweenness_centrality(G, weight='abs_weight', normalized=True) if G.number_of_edges() else {n: 0 for n in G.nodes}
    bridge = {n: 0.0 for n in G.nodes}
    bridge_n = {n: 0 for n in G.nodes}
    for u, v, d in G.edges(data=True):
        if domains.get(u) != domains.get(v):
            bridge[u] += d['abs_weight']; bridge[v] += d['abs_weight']
            bridge_n[u] += 1; bridge_n[v] += 1
    out = pd.DataFrame({'feature': list(G.nodes), 'domain': [domains.get(n) for n in G.nodes],
                         'degree': [degree[n] for n in G.nodes], 'strength': [strength[n] for n in G.nodes],
                         'betweenness': [between[n] for n in G.nodes],
                         'bridge_degree': [bridge_n[n] for n in G.nodes],
                         'bridge_strength': [bridge[n] for n in G.nodes]})
    return out.sort_values(['bridge_strength', 'strength'], ascending=False), G


model, alpha_table, selected_alpha = select_glasso_ebic_density_constrained(Z)
alpha_table.to_csv(OUT / 'complete_case_alpha_selection_ebic.csv', index=False)
print(f'Selected alpha = {selected_alpha}')
alpha_table


## 3. Fit the final network, compute edges, save partial-correlation outputs

In [ ]:
prec = model.precision_
D = np.sqrt(np.diag(prec))
P = -prec / np.outer(D, D)
np.fill_diagonal(P, 1)

edges = edge_df_from_partial(P, cols)
edges['source_domain'] = edges.source.map(DOM)
edges['target_domain'] = edges.target.map(DOM)
edges['cross_domain'] = edges.source_domain != edges.target_domain
edges.to_csv(OUT / 'complete_case_partial_edges.csv', index=False)
pd.DataFrame(P, index=cols, columns=cols).to_csv(OUT / 'complete_case_partial_correlation_matrix.csv')

p = len(cols)
summary = {
    'n_participants_complete_case': int(n_complete),
    'n_participants_primary_imputed': 1286,
    'pct_of_primary_cohort_retained': round(100 * n_complete / 1286, 1),
    'n_primary_direct_biomarkers': int(p),
    'selection_method': 'EBIC gamma=0.50 with pre-specified sparse interpretability constraint density <= 0.20 (identical rule to primary network)',
    'selected_alpha': float(selected_alpha),
    'n_edges': int(len(edges)),
    'network_density': float(len(edges) / (p * (p - 1) / 2)),
    'n_cross_domain_edges': int(edges.cross_domain.sum()),
    'note': 'Complete-case sensitivity network: rows with any missing value among the 29 primary biomarkers were excluded (no imputation). Same preprocessing, z-scoring, Graphical LASSO EBIC(gamma=0.5)+density<=0.20 selection rule, and centrality/bridge-centrality formulas as the primary network.'
}
(OUT / 'complete_case_model_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
summary


## 4. Node-centrality and bridge-centrality metrics

In [ ]:
metrics, G = graph_metrics(edges, DOM)
metrics.to_csv(OUT / 'Table_complete_case_centrality_bridge_metrics.csv', index=False)
metrics.head(10).to_csv(OUT / 'top_bridge_biomarkers_complete_case.csv', index=False)
metrics.sort_values('strength', ascending=False).head(10).to_csv(OUT / 'top_central_biomarkers_complete_case.csv', index=False)
metrics.head(10)


## 5. Figures

In [ ]:
plt.figure(figsize=(11, 9))
Gp = nx.Graph()
for f in cols:
    Gp.add_node(f, domain=DOM[f])
for _, r in edges.iterrows():
    Gp.add_edge(r.source, r.target, weight=r.weight, abs_weight=r.abs_weight)
pos = nx.spring_layout(Gp, seed=SEED, k=0.55)
widths = [max(0.5, 4 * Gp[u][v]['abs_weight']) for u, v in Gp.edges]
nx.draw_networkx_edges(Gp, pos, width=widths, alpha=.45)
nx.draw_networkx_nodes(Gp, pos, node_size=650)
nx.draw_networkx_labels(Gp, pos, font_size=8)
plt.axis('off'); plt.tight_layout()
plt.savefig(OUT / 'Figure_complete_case_partial_network.png', dpi=200)
plt.close()

for col_, fn, title in [('bridge_strength', 'Figure_top_bridge_strength_complete_case.png', 'Top bridge biomarkers (complete-case)'),
                         ('strength', 'Figure_top_node_strength_complete_case.png', 'Top central biomarkers (complete-case)')]:
    top = metrics.sort_values(col_, ascending=False).head(12).iloc[::-1]
    plt.figure(figsize=(8, 5))
    plt.barh(top.feature, top[col_])
    plt.title(title)
    plt.tight_layout()
    plt.savefig(OUT / fn, dpi=200)
    plt.close()
print('Figures saved to', OUT)


## 6. Concordance with the primary (median-imputed, n=1,286) network

Quantifies agreement in: (a) retained edges (Jaccard similarity / % of primary edges preserved), (b) rank concordance of node strength and bridge strength (Spearman correlation across the 29 shared biomarkers), and (c) overlap of the top-8 bridge biomarkers.

In [ ]:
primary_edges = pd.read_csv(PROJECT_ROOT / 'outputs' / '04_primary_graphical_lasso' / 'primary_graphical_lasso_partial_edges.csv')
primary_metrics = pd.read_csv(PROJECT_ROOT / 'outputs' / '05_centrality_bridge' / 'Table_primary_centrality_bridge_metrics.csv')
primary_summary = json.loads((PROJECT_ROOT / 'outputs' / '04_primary_graphical_lasso' / 'primary_graphical_lasso_model_summary.json').read_text())

def edge_set(df):
    return set(tuple(sorted((r.source, r.target))) for r in df.itertuples())

es_primary = edge_set(primary_edges)
es_cc = edge_set(edges)
shared = es_primary & es_cc
jaccard = len(shared) / len(es_primary | es_cc) if (es_primary | es_cc) else np.nan
pct_primary_edges_preserved = len(shared) / len(es_primary) * 100 if es_primary else np.nan

merged = primary_metrics[['feature', 'bridge_strength', 'strength']].merge(
    metrics[['feature', 'bridge_strength', 'strength']], on='feature', suffixes=('_primary', '_cc'))
bridge_spearman = merged['bridge_strength_primary'].corr(merged['bridge_strength_cc'], method='spearman')
strength_spearman = merged['strength_primary'].corr(merged['strength_cc'], method='spearman')

top8_primary = set(primary_metrics.sort_values('bridge_strength', ascending=False).head(8).feature)
top8_cc = set(metrics.sort_values('bridge_strength', ascending=False).head(8).feature)

comparison = {
    'primary_n': 1286,
    'complete_case_n': int(n_complete),
    'primary_alpha': primary_summary['selected_alpha'],
    'complete_case_alpha': float(selected_alpha),
    'primary_n_edges': primary_summary['n_edges'],
    'complete_case_n_edges': int(len(edges)),
    'primary_density': primary_summary['network_density'],
    'complete_case_density': float(len(edges) / (p * (p - 1) / 2)),
    'n_shared_edges': len(shared),
    'jaccard_edge_similarity': jaccard,
    'pct_primary_edges_preserved_in_complete_case': pct_primary_edges_preserved,
    'bridge_strength_spearman_correlation': bridge_spearman,
    'node_strength_spearman_correlation': strength_spearman,
    'top8_bridge_biomarkers_primary': sorted(top8_primary),
    'top8_bridge_biomarkers_complete_case': sorted(top8_cc),
    'top8_overlap': sorted(top8_primary & top8_cc),
    'top8_overlap_count': len(top8_primary & top8_cc),
    'top8_only_in_primary': sorted(top8_primary - top8_cc),
    'top8_only_in_complete_case': sorted(top8_cc - top8_primary),
}
(OUT / 'complete_case_vs_primary_comparison.json').write_text(json.dumps(comparison, indent=2), encoding='utf-8')
comparison


## 7. Summary for the manuscript

Report the actual numbers from `complete_case_model_summary.json` and `complete_case_vs_primary_comparison.json` above verbatim in the Results / Robustness section — do not restate qualitative claims (e.g. "highly concordant") without the supporting statistics produced here.